# 07 — Curation Report

## Objective

Consolidate stages 01–06 into a review queue and a curated dataset manifest while keeping human decisions in a dedicated log.

## Motivation

Objective findings and automated review signals must remain distinguishable. A stable `candidate_key` preserves previous human decisions when the queue is regenerated, and raw data is never modified.

## Inputs

- Manifests from stages 01–06.
- Canonical artifacts declared by those stages.
- Existing `curation_log.csv`, when present.

## Outputs

- `curation/outputs/07-curation-report/curation_queue.csv`
- `curation/outputs/07-curation-report/curation_log.csv`
- `curation/outputs/07-curation-report/curated_manifest.csv`
- `curation/outputs/07-curation-report/manifest.yaml`


In [ ]:
# Environment-specific setup
import pathlib
import sys

if 'google.colab' not in str(get_ipython()):
    base_folder = pathlib.Path('../../../')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')

ROOT = base_folder.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
from curation.common import load_config, prepare_dataset, stage_output_dir

CONFIG = load_config(ROOT)
DATASET = prepare_dataset(ROOT)
IMAGES_DIR = DATASET["images_dir"]
ANNOTATIONS_DIR = DATASET["annotations_dir"]

from curation.common import load_manifest, stable_key, write_manifest
import pandas as pd

STAGE = "07-curation-report"
STAGE_DIR = stage_output_dir(STAGE, ROOT)
STAGE_NAMES = ["01-dataset-audit", "02-annotation-audit", "03-class-distribution", "04-duplicate-analysis", "05-image-quality", "06-active-label-cleaning"]
manifests = {stage: load_manifest(stage, ROOT) for stage in STAGE_NAMES}
missing = [stage for stage, manifest in manifests.items() if not manifest]
if missing:
    raise FileNotFoundError(f"Manifests ausentes: {missing}")

def artifact(stage, name):
    return stage_output_dir(stage, ROOT, create=False) / name

inventory_path = artifact("01-dataset-audit", "inventory.csv")
annotation_findings_path = artifact("02-annotation-audit", "annotation_findings.csv")
exact_duplicates_path = artifact("04-duplicate-analysis", "exact_duplicates.csv")
duplicate_candidates_path = artifact("04-duplicate-analysis", "duplicate_review_candidates.csv")
quality_outliers_path = artifact("05-image-quality", "image_quality_outliers.csv")
alc_candidates_path = artifact("06-active-label-cleaning", "alc_label_candidates.csv")

In [ ]:
inventory = pd.read_csv(inventory_path)
annotation_findings = pd.read_csv(annotation_findings_path)
exact_duplicates = pd.read_csv(exact_duplicates_path)
duplicate_candidates = pd.read_csv(duplicate_candidates_path)
quality_outliers = pd.read_csv(quality_outliers_path)
alc_candidates = pd.read_csv(alc_candidates_path, dtype={"original_label": str, "predicted_label": str})

queue_rows = []
def add(source, item_a, *, item_b="", reason="", score=None, interpretation="manual_review_required", source_key=None):
    key = str(source_key) if source_key is not None else stable_key(source, item_a, item_b, reason)
    queue_rows.append({"candidate_key": key, "source": source, "item_a": item_a, "item_b": item_b, "reason": reason, "score": score, "automatic_interpretation": interpretation})

for row in inventory[~inventory["readable"]].itertuples(index=False):
    add("dataset_integrity", row.relative_path, reason="unreadable_image", interpretation="objective_finding")
for row in inventory[inventory["annotation_relative_path"].isna()].itertuples(index=False):
    add("dataset_integrity", row.relative_path, reason="image_without_unique_annotation", interpretation="objective_finding")
for row in annotation_findings.itertuples(index=False):
    add("annotation_audit", row.relative_path, reason=row.finding_type, interpretation="objective_finding" if row.severity == "error" else "manual_review_required", source_key=f"{row.annotation_relative_path}:{row.shape_index}:{row.finding_type}")
for digest, group in exact_duplicates.groupby("sha256"):
    paths = sorted(group["relative_path"].tolist())
    for duplicate in paths[1:]:
        add("exact_duplicate", paths[0], item_b=duplicate, reason="same_sha256", interpretation="objective_finding", source_key=f"{digest}:{paths[0]}:{duplicate}")
for row in duplicate_candidates.itertuples(index=False):
    add("near_duplicate", row.image_a, item_b=row.image_b, reason="perceptual_similarity", score=row.ssim_256_gray, source_key=f"{row.image_a}:{row.image_b}")
for row in quality_outliers.itertuples(index=False):
    add("image_quality", row.relative_path, reason=row.metric, score=row.robust_z, source_key=f"{row.relative_path}:{row.metric}")
for row in alc_candidates.itertuples(index=False):
    add("alc_label_candidate", row.relative_path, reason=f"original={row.original_label};predicted={row.predicted_label}", score=row.alc_phi, source_key=row.candidate_key)

queue = pd.DataFrame(queue_rows).sort_values(["automatic_interpretation", "source", "score"], ascending=[True, True, False], na_position="last").reset_index(drop=True)
queue_path = STAGE_DIR / "curation_queue.csv"
queue.to_csv(queue_path, index=False)
display(queue.head(30))

In [ ]:
log_path = STAGE_DIR / "curation_log.csv"
human_columns = ["manual_status", "manual_action", "notes"]
base_log = queue.copy()
base_log["manual_status"] = "pending"
base_log["manual_action"] = ""
base_log["notes"] = ""
if log_path.is_file():
    previous = pd.read_csv(log_path, keep_default_na=False)
    if previous["candidate_key"].duplicated().any():
        raise ValueError("curation_log.csv contains duplicate candidate_key values.")
    preserved = previous.set_index("candidate_key")[human_columns]
    base_log = base_log.set_index("candidate_key")
    common_keys = base_log.index.intersection(preserved.index)
    base_log.loc[common_keys, human_columns] = preserved.loc[common_keys, human_columns]
    base_log = base_log.reset_index()
base_log.to_csv(log_path, index=False)

remove_paths = set()
for row in base_log[base_log["manual_action"].isin(["remove", "merge_duplicate"])].itertuples(index=False):
    if row.source != "alc_label_candidate":
        if isinstance(row.item_a, str) and row.item_a:
            remove_paths.add(row.item_a)
        if row.manual_action == "merge_duplicate" and isinstance(row.item_b, str) and row.item_b:
            remove_paths.add(row.item_b)

curated = inventory.copy()
curated["keep_after_manual_curation"] = ~curated["relative_path"].isin(remove_paths)
curated_path = STAGE_DIR / "curated_manifest.csv"
curated.to_csv(curated_path, index=False)

In [ ]:
write_manifest(
    STAGE,
    "07-curation-report.ipynb",
    inputs={f"manifest_{stage[:2]}": stage_output_dir(stage, ROOT, create=False) / "manifest.yaml" for stage in STAGE_NAMES} | {"inventory": inventory_path, "annotation_findings": annotation_findings_path, "exact_duplicates": exact_duplicates_path, "duplicate_candidates": duplicate_candidates_path, "quality_outliers": quality_outliers_path, "alc_candidates": alc_candidates_path},
    parameters={"human_decision_key": "candidate_key", "raw_data_mutation": False},
    artifacts=[queue_path, log_path, curated_path],
    summary={"queue_items": int(len(queue)), "objective_findings": int((queue["automatic_interpretation"] == "objective_finding").sum()), "review_candidates": int((queue["automatic_interpretation"] == "manual_review_required").sum()), "manual_status": {str(k): int(v) for k, v in base_log["manual_status"].value_counts().sort_index().items()}, "images_kept": int(curated["keep_after_manual_curation"].sum()), "images_removed": int((~curated["keep_after_manual_curation"]).sum())},
    repo_root=ROOT,
)
gitkeep = STAGE_DIR / ".gitkeep"
if gitkeep.exists():
    gitkeep.unlink()